In [ ]:
import numpy as np
import requests
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import time
from tqdm.notebook import tqdm
import os
import gc
import matplotlib.colors as colors

In [ ]:
# Ensure "Datos_Galaxia_117255.npz" is present in your local path before execution
data = np.load("Datos_Galaxia_117255.npz", allow_pickle=True)
print(data.files)

In [ ]:
Xs = data["X"]
Ys = data["Y"]
Zs = data["Z"]

Bxs = data["Bx"]
Bys = data["By"]
Bzs = data["Bz"]

Mass = data["Mass"]
rhos = data["Density"]
Us = data["Internal_Energy"]
Lambdas = data["GFM_CoolingRate"]
e = data["Electron_Abundance"]
H_Ne = data["Neutral_H_Abundance"]
Phi = data["Potential"]

ID = data["SubfindID"]
SubID = data["SubhaloID"]
Snap = data["SN"]

In [ ]:
Posx = data["SubhaloPosx"]
Posy = data["SubhaloPosy"]
Posz = data["SubhaloPosz"]

CMx = data["SubhaloCMx"]
CMy = data["SubhaloCMy"]
CMz = data["SubhaloCMz"]

## Figures

### X vs Y

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  x_centrado = Xs[i] - CMx[i]
  y_centrado = Ys[i] - CMy[i]

  ax.plot(x_centrado,y_centrado,".", markersize = 0.1)
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel("X", fontsize=8)
  ax.set_ylabel("Y", fontsize=8)
  ax.set_xlim(-400,400)
  ax.set_ylim(-250,250)

plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

In [ ]:
densidades_concatenadas = np.concatenate(rhos)
rho_min = densidades_concatenadas.min()
rho_max = densidades_concatenadas.max()

fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
    fila = i // 5
    columna = i % 5
    ax = axs[fila, columna]

    densidad = rhos[i]
    x_centrado = Xs[i] - CMx[i]
    y_centrado = Ys[i] - CMy[i]

    idx_orden = np.argsort(densidad)
    x_graficar = x_centrado[idx_orden]
    y_graficar = y_centrado[idx_orden]
    densidad_graficar = densidad[idx_orden]

    sc = ax.scatter(x_graficar, y_graficar,
                    c=densidad_graficar,
                    cmap='jet',
                    s=0.1,
                    alpha=0.5,
                    norm=colors.LogNorm(vmin=rho_min, vmax=rho_max))

    ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
    ax.set_xlabel("X", fontsize=8)
    ax.set_ylabel("Y", fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    ax.set_xlim(-400, 400)
    ax.set_ylim(-250, 250)

plt.tight_layout(rect=[0, 0, 0.9, 1])


cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label(r"Densidad de Gas ($\log_{10}\rho_{gas}$)", fontsize=12, labelpad=10)

plt.show()

### $\Lambda$ vs $\rho$

(Colling Rate vs Density)

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(rhos[i],Lambdas[i],".", markersize = 0.1)
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel(r"$\rho$", fontsize=8)
  ax.set_ylabel(r"$\Lambda$", fontsize=8, rotation=0)
  ax.set_xlim(1.3e-9,0.9)
  ax.set_ylim(-1e-21,0.25e-21)
  ax.set_xscale('log')


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### U vs $\rho$

(Internal Energy vs Density)

In [ ]:
maxi = np.concatenate(Us)
maximU = maxi.max()
minmU = maxi.min()

In [ ]:
aver = []
aver_mass = []
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  U_mass = np.sum(Mass[i] * Us[i]) / np.sum(Mass[i])
  ax = axs[fila, columna]
  ax.plot(rhos[i],Us[i],".", markersize = 0.1)
  ax.axhline(3049.8975, color='r', linestyle='--', linewidth=1, label='Media datos')
  ax.axhline((3049.8975+10**(2.78))/2, color='y', linestyle='--', linewidth=1, label='Media de corte')
  ax.axhline(10**(2.78), color='g', linestyle='--', linewidth=1, label='Cardona & Muñoz Cuartas: 2.78')
  aver.append(np.mean(Us[i]))
  aver_mass.append(U_mass)
  ax.set_xscale('log')
  ax.set_yscale('log')
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel(r"$\rho$", fontsize=8)
  ax.set_ylabel("U", fontsize=8)
  ax.set_xlim(1.3e-9,0.9)
  ax.set_ylim(minmU,maximU+0.1*maximU)
  ax.legend()

plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

In [ ]:
print(np.mean(aver),np.mean(aver_mass),10**2.43)

### $H^+$ vs  U

(Neutral Hydrogen Abundance  vs Internal Energy)

In [ ]:
maxi = np.concatenate(H_Ne)
maximH = maxi.max()
minmH = maxi.min()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(Us[i],H_Ne[i],".", markersize = 0.1)
  ax.set_xscale('log')
  ax.set_yscale('log')
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel("U", fontsize=8)
  ax.set_ylabel(r"$H^+$", fontsize=8)
  ax.set_xlim(minmU,maximU+0.5*maximU)
  ax.set_ylim(minmH-0.1*minmH, maximH+0.8*maximH)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### $e^-$ vs U

(Electron Abundance  vs Internal Energy)

In [ ]:
maxi = np.concatenate(e)
maxime = maxi.max()
minme = maxi.min()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(Us[i],e[i],".", markersize = 0.1)
  ax.set_xscale('log')
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel("U", fontsize=8)
  ax.set_ylabel(r"$e^-$", fontsize=8)
  ax.set_xlim(minmU,maximU+0.5*maximU)
  ax.set_ylim(minme-0.1, maxime+0.1*maxime)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### |$\vec{B}$| vs $\Phi$

(magnetic field magnitude vs Potential)

In [ ]:
B_Mags = []

for i in range(20):
  B_Mags.append(np.sqrt(Bxs[i]**2 + Bys[i]**2 + Bzs[i]**2))


In [ ]:
maxi = np.concatenate(Phi)
maximPhi = maxi.max()
minmPhi = maxi.min()
maxi = np.concatenate(B_Mags)
maximB = maxi.max()
minmB = maxi.min()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  B_Mag = np.sqrt(Bxs[i]**2 + Bys[i]**2 + Bzs[i]**2)
  ax.plot(Phi[i],B_Mag,".", markersize = 0.1)
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel(r"$\Phi$", fontsize=8)
  ax.set_ylabel(r"|$\vec{B}$|", fontsize=8)
  ax.set_yscale('log')
  ax.set_xlim(minmPhi,maximPhi)
  ax.set_ylim(minmB, maximB)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

## Cut into U = 2.78

In [ ]:
Mass_cut = []
xs_cut = []
ys_cut = []
zs_cut = []
Lambdas_cut = []
rhos_cut = []
U_cut = []
H_Ne_cut = []
e_cut =[]
Bxs_cut = []
Bys_cut = []
Bzs_cut = []
Phi_cut = []

In [ ]:
for i in range(20):
  U_mask = Us[i] <= 10**2.78

  Mass_cut.append(Mass[i][U_mask])
  xs_cut.append(Xs[i][U_mask])
  ys_cut.append(Ys[i][U_mask])
  zs_cut.append(Zs[i][U_mask])
  Lambdas_cut.append(Lambdas[i][U_mask])
  rhos_cut.append(rhos[i][U_mask])
  U_cut.append(Us[i][U_mask])
  H_Ne_cut.append(H_Ne[i][U_mask])
  e_cut.append(e[i][U_mask])
  Bxs_cut.append(Bxs[i][U_mask])
  Bys_cut.append(Bys[i][U_mask])
  Bzs_cut.append(Bzs[i][U_mask])
  Phi_cut.append(Phi[i][U_mask])

In [ ]:
Mass_cut = np.array(Mass_cut, dtype=object)
xs_cut = np.array(xs_cut, dtype=object)
ys_cut = np.array(ys_cut, dtype=object)
zs_cut = np.array(zs_cut, dtype=object)
Lambdas_cut = np.array(Lambdas_cut, dtype=object)
rhos_cut = np.array(rhos_cut, dtype=object)
U_cut = np.array(U_cut, dtype=object)
H_Ne_cut = np.array(H_Ne_cut, dtype=object)
e_cut = np.array(e_cut, dtype=object)
Bxs_cut = np.array(Bxs_cut, dtype=object)
Bys_cut = np.array(Bys_cut, dtype=object)
Bzs_cut = np.array(Bzs_cut, dtype=object)
Phi_cut = np.array(Phi_cut, dtype=object)

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(rhos_cut[i],U_cut[i],".", markersize = 0.1)
  aver.append(np.mean(Us[i]))
  aver_mass.append(U_mass)
  ax.set_xscale('log')
  ax.set_yscale('log')
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel(r"$\rho$", fontsize=8)
  ax.set_ylabel("U", fontsize=8)
  ax.set_xlim(1.3e-9,0.9)
  ax.set_ylim(minmU,maximU+0.1*maximU)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### X vs Y

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  x_centrado = xs_cut[i] - CMx[i]
  y_centrado = ys_cut[i] - CMy[i]

  ax.plot(x_centrado,y_centrado,".", markersize = 0.1)
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel("X", fontsize=8)
  ax.set_ylabel("Y", fontsize=8)
  ax.set_xlim(-400,400)
  ax.set_ylim(-250,250)

plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

In [ ]:
densidades_concatenadas = np.concatenate(rhos_cut)
rho_min = densidades_concatenadas.min()
rho_max = densidades_concatenadas.max()

fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
    fila = i // 5
    columna = i % 5
    ax = axs[fila, columna]

    densidad = rhos_cut[i]
    x_centrado = xs_cut[i] - CMx[i]
    y_centrado = ys_cut[i] - CMy[i]

    idx_orden = np.argsort(densidad)
    x_graficar = x_centrado[idx_orden]
    y_graficar = y_centrado[idx_orden]
    densidad_graficar = densidad[idx_orden]

    sc = ax.scatter(x_graficar, y_graficar,
                    c=densidad_graficar,
                    cmap='jet',
                    s=0.1,
                    alpha=0.5,
                    norm=colors.LogNorm(vmin=rho_min, vmax=rho_max))

    ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
    ax.set_xlabel("X", fontsize=8)
    ax.set_ylabel("Y", fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    ax.set_xlim(-400, 400)
    ax.set_ylim(-250, 250)

plt.tight_layout(rect=[0, 0, 0.9, 1])


cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sc, cax=cax)
cbar.set_label(r"Densidad de Gas ($\log_{10}\rho_{gas}$)", fontsize=12, labelpad=10)

plt.show()

### $\Lambda$ vs $\rho$

(Colling Rate vs Density)

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(rhos_cut[i],Lambdas_cut[i],".", markersize = 0.1)
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel(r"$\rho$", fontsize=8)
  ax.set_ylabel(r"$\Lambda$", fontsize=8, rotation=0)
  ax.set_xlim(1.3e-9,0.9)
  ax.set_ylim(-1e-21,0.25e-21)
  ax.set_xscale('log')


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### $H^+$ vs  U

(Neutral Hydrogen Abundance  vs Internal Energy)

In [ ]:
maxi = np.concatenate(H_Ne)
maximH = maxi.max()
minmH = maxi.min()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(U_cut[i],H_Ne_cut[i],".", markersize = 0.1)
  ax.set_xscale('log')
  ax.set_yscale('log')
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel("U", fontsize=8)
  ax.set_ylabel(r"$H^+$", fontsize=8)
  ax.set_xlim(minmU,maximU+0.5*maximU)
  ax.set_ylim(minmH-0.1*minmH, maximH+0.8*maximH)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### $e^-$ vs U

(Electron Abundance  vs Internal Energy)

In [ ]:
maxi = np.concatenate(e)
maxime = maxi.max()
minme = maxi.min()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  ax.plot(U_cut[i],e_cut[i],".", markersize = 0.1)
  ax.set_xscale('log')
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel("U", fontsize=8)
  ax.set_ylabel(r"$e^-$", fontsize=8)
  ax.set_xlim(minmU,maximU+0.5*maximU)
  ax.set_ylim(minme-0.1, maxime+0.1*maxime)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()

### |$\vec{B}$| vs $\Phi$

(magnetic field magnitude vs Potential)

In [ ]:
B_Mags = []

for i in range(20):
  B_Mags.append(np.sqrt(Bxs_cut[i]**2 + Bys_cut[i]**2 + Bzs_cut[i]**2))


In [ ]:
maxi = np.concatenate(Phi)
maximPhi = maxi.max()
minmPhi = maxi.min()
maxi = np.concatenate(B_Mags)
maximB = maxi.max()
minmB = maxi.min()

In [ ]:
fig, axs = plt.subplots(4, 5, figsize=(18, 12))

for i in range(20):
  fila = i // 5
  columna = i % 5

  ax = axs[fila, columna]
  B_Mag = np.sqrt(Bxs_cut[i]**2 + Bys_cut[i]**2 + Bzs_cut[i]**2)
  ax.plot(Phi_cut[i],B_Mag,".", markersize = 0.1)
  ax.set_title(f"Galaxia {SubID[i]} y Snap {Snap[i]}", fontsize=10)
  ax.set_xlabel(r"$\Phi$", fontsize=8)
  ax.set_ylabel(r"|$\vec{B}$|", fontsize=8)
  ax.set_yscale('log')
  ax.set_xlim(minmPhi,maximPhi)
  ax.set_ylim(minmB, maximB)


plt.tight_layout()
fig.subplots_adjust(wspace=0.4, hspace=0.4)
plt.show()